# DiffSinger Miadu Fine-tuning (推理試聽版)
**已配置**：
- **100步存檔**：每 100 步保存一個 Checkpoint。
- **斷點續練**：重啟後自動接續進度。
- **推理試聽**：可以直接測試訓練成果。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取/更新代碼與環境配置

In [ ]:
%cd /content/
import os
if not os.path.exists('diffsinger-repo'):
    !git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
else:
    %cd diffsinger-repo
    !git pull
    %cd ..

%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：數據同步與路徑對齊

In [ ]:
import os
drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"

target_link = "/content/diffsinger-repo/checkpoints"
drive_target = f"{drive_root}/checkpoints_finetune"
!mkdir -p "{drive_target}"

if os.path.exists(target_link) and not os.path.islink(target_link):
    !cp -rv "{target_link}"/* "{drive_target}/" 2>/dev/null || true
    !rm -rf "{target_link}"

if not os.path.exists(target_link):
    !ln -s "{drive_target}" "{target_link}"

def robust_sync(src_dir, dest_path, min_size_mb=1):
    dest_dir = os.path.dirname(dest_path)
    !mkdir -p "{dest_dir}"
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < min_size_mb * 1024 * 1024:
        !cp -rv "{src_dir}" "{dest_dir}/.."

robust_sync(f"{drive_root}/1117_opencpop_ds1000_strict_pinyin", 
            "checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 800)
robust_sync(f"{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02", 
            "checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_160000.ckpt", 10)
robust_sync(f"{drive_root}/miadu_finetune", 
            "data/binary/miadu_finetune/train.data", 1000)
print('| 數據環境已就緒。')

## 第四步：啟動訓練 (微調)

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1

## 第五步：推理試聽 (測試音色)

In [ ]:
import sys
import os
import torch
import IPython.display as ipd
from utils.hparams import hparams
from inference.svs.ds_e2e import DiffSingerE2EInfer

# --- 設定區 ---
CHECKPOINT_STEP = 700  # 請輸入您想測試的步數
TEST_TEXT = "小酒窩長睫毛，是你最美的記號"
TEST_NOTES = "C#4 | F#4 | G#4 | A#4 F#4 | F#4 C#4 | C#4 | rest | C#4 | A#4 | G#4"
TEST_DURATIONS = "0.4 | 0.3 | 0.2 | 0.5 0.1 | 0.3 0.2 | 0.3 | 0.2 | 0.3 | 0.3 | 0.3"
# ------------

os.environ['PYTHONPATH'] = "."
sys.argv = [
    'inference/svs/ds_e2e.py', 
    '--config', 'usr/configs/midi/e2e/miadu_finetune.yaml', 
    '--exp_name', 'miadu_finetune_v1'
]
from utils.hparams import set_hparams
set_hparams()

# 強制載入指定步數的權重
hparams['work_dir'] = 'checkpoints/miadu_finetune_v1'
target_ckpt = f"checkpoints/miadu_finetune_v1/model_ckpt_steps_{CHECKPOINT_STEP}.ckpt"
if not os.path.exists(target_ckpt):
    print(f"❌ 找不到權重檔: {target_ckpt}")
else:
    print(f"✅ 正在載入權重: {target_ckpt}")
    infer_ins = DiffSingerE2EInfer(hparams)
    
    # 手動覆蓋載入路徑
    from utils import load_ckpt
    load_ckpt(infer_ins.model, target_ckpt, 'model')
    
    inp = {
        'text': TEST_TEXT,
        'notes': TEST_NOTES,
        'notes_duration': TEST_DURATIONS,
        'input_type': 'word'
    }
    
    with torch.no_grad():
        wav = infer_ins.forward_model(inp)
        
    print(f"| 推理完成！正在生成播放器...")
    ipd.display(ipd.Audio(wav, rate=hparams['audio_sample_rate']))